# Route A — TF-IDF / BoW / One-Hot Encoders × Many Classifiers

This notebook extends the original Route A with three additions you asked for:

1. **Extensive preprocessing** (the group's rich-cleaning pipeline from `Preprocessing.ipynb`)
   compared head-to-head against the light EDA cleaning, *on the same split*.
2. **A single leaderboard** over **preprocessing × vectorizer × classifier**, scored with
   **stratified cross-validation** — vectorizers: Bag-of-Words (counts), One-Hot (binary),
   TF-IDF; classifiers: ComplementNB, LinearSVC, a non-linear **SVC (RBF)**, LogisticRegression,
   and RandomForest.
3. A **source-prediction section**: the deployed sentiment model is *source-blind* (the private
   test set has no Review/Tweet label), but we train a separate classifier that recovers the
   source from text and use it to validate the per-domain sentiment breakdown without ever
   needing the true `Type`.

Everything loads the one shared split written by `EDA.ipynb`.

### 1/ Imports and configuration

In [3]:
import os, re, html, pickle, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

from collections import Counter
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, classification_report, confusion_matrix

nltk.download("stopwords", quiet=True)
nltk.download("wordnet",   quiet=True)
nltk.download("omw-1.4",   quiet=True)

SEED       = 42
MODEL_OUT  = "route_a.model"
N_SPLITS   = 5            # CV folds for the leaderboard
RUN_RBF    = True        # set False to skip the slow non-linear SVC / RandomForest rows
np.random.seed(SEED)

### 2/ Load the shared split

`EDA.ipynb` must be re-run with the patched save cell (see the note in chat) so that
`data_split.csv` carries the **raw `Text`** column. Extensive preprocessing needs the raw
text — `clean_eda` has already removed URLs/@mentions/HTML and lowercased, which is exactly
the signal the rich pipeline wants to keep. If `Text` is absent the notebook still runs, but
the extensive-preprocessing rows fall back to light cleaning and the comparison is moot.

In [4]:
data = pd.read_csv("data_split.csv", index_col=0)
data["clean_eda"] = data["clean_eda"].fillna("").astype(str)
data["sentiment"] = data["sentiment"].astype(int)

RAW_COL = next((c for c in data.columns if c.lower() in ("text", "rawtext", "raw_text")), None)
HAVE_RAW = RAW_COL is not None
if not HAVE_RAW:
    print("[warn] no raw Text column in data_split.csv -> extensive preprocessing will fall "
          "back to clean_eda. Re-run the patched EDA save cell to enable the real comparison.")
    data["Text"] = data["clean_eda"]
    RAW_COL = "Text"

print("Loaded:", len(data), "documents | raw column:", RAW_COL)
print(data["split"].value_counts().to_string())
print("\nClass balance overall:")
print(data["sentiment"].value_counts(normalize=True).sort_index().round(3).to_string())

Loaded: 255083 documents | raw column: Text
split
train    178557
val       38263
test      38263

Class balance overall:
sentiment
-1    0.352
 0    0.249
 1    0.399


### 3/ Two preprocessing policies, one pickle-safe transformer

`RouteAPreprocessor` is a **top-level** estimator (no lambdas, no closures) so it pickles
cleanly into the `.model` file later.

* `mode="light"` reproduces the EDA cleaning (regex only — no extra runtime dependencies).
* `mode="rich"` is the extensive pipeline: strip HTML, map URLs/@mentions to `URL`/`MNTN`,
  demojize, keep ALL-CAPS as an emphasis signal, compress elongations, prune stopwords while
  **keeping negations/intensifiers**, and apply a frozen fuzzy spelling-correction map.

The correction map is the only data-driven part and is fit **on the training set only**; at
inference it is a plain dict lookup, so `rapidfuzz` is *not* needed in deployment (only
`emoji` + `beautifulsoup4`).

In [5]:
# light cleaning (matches EDA.clean_eda) ------------------------------------------------
_url_re   = re.compile(r"http\S+|www\.\S+")
_ment_re  = re.compile(r"@\w+")
_html_re  = re.compile(r"<[^>]+>")
_sp_re    = re.compile(r"\s+")
_elong_re = re.compile(r"(.)\1{2,}")

def clean_light(text: str) -> str:
    text = str(text)
    text = html.unescape(text)
    text = _html_re.sub(" ", text)
    text = _url_re.sub(" ", text)
    text = _ment_re.sub(" ", text)
    text = text.replace("#", " ")
    return _sp_re.sub(" ", text.lower()).strip()

# rich stopword set: prune sklearn's list but KEEP sentiment-bearing function words --------
from sklearn.feature_extraction import _stop_words
_base        = set(_stop_words.ENGLISH_STOP_WORDS)
_negations   = {"no","not","never","neither","none","nothing","nobody","nowhere","nor",
                "without","cannot","n't","n\u2019t","n\u2018t"}
_intensifiers= {"very","really","extremely","incredibly","highly","remarkably","exceptionally",
                "enormously","immensely","tremendously","significantly","seriously","genuinely",
                "truly","deeply","strongly","greatly","super","totally","absolutely","completely",
                "utterly","ridiculously","insanely","unbelievably","awfully","terribly","pretty",
                "quite","rather","real","mega","crazy","entirely","perfectly","thoroughly",
                "wholly","downright","positively","decidedly"}
_downtoners  = {"slightly","somewhat","fairly","rather","quite","relatively","moderately","mildly",
                "partly","almost","nearly","just","barely","hardly","scarcely","somehow","loosely","vaguely"}
_other       = {"one","five","few","however","though","nevertheless","but","although","too","only",
                "just","still","already","more","less","do"}
STOP_RICH    = _base - (_intensifiers | _downtoners | _negations | _other)


class RouteAPreprocessor(BaseEstimator, TransformerMixin):
    """Pickle-safe text cleaner. mode in {'light','rich'}."""
    def __init__(self, mode="light", correction_map=None, stop=None):
        self.mode = mode
        self.correction_map = correction_map if correction_map is not None else {}
        self.stop = set(stop) if stop is not None else set(STOP_RICH)

    def fit(self, X, y=None):
        return self

    def _rich(self, x):
        from bs4 import BeautifulSoup          # imported lazily so 'light' needs no deps
        import emoji
        x = str(x)
        x = BeautifulSoup(x, "html.parser").get_text()
        x = _url_re.sub(" URL ", x)
        x = _ment_re.sub(" MNTN ", x)
        x = emoji.demojize(x, delimiters=(" ", " ")).replace("_", " ")
        x = re.sub(r"[^\w\s]", " ", x)
        x = _sp_re.sub(" ", x)
        toks = [t for t in x.split() if t.lower() not in self.stop and len(t) >= 2]
        toks = [self.correction_map.get(t, t) for t in toks]
        toks = [t if (t.isupper() and len(t) > 1) else t.lower() for t in toks]
        toks = [_elong_re.sub(r"\1\1", t) for t in toks]
        return " ".join(toks)

    def transform(self, X):
        fn = self._rich if self.mode == "rich" else clean_light
        return [fn(t) for t in X]

### 4/ Fit the spelling-correction map (TRAIN ONLY) and build cleaned columns

`build_correction_map` is the conservative, capped fuzzy corrector from `route_nbsvm`. It is
fit only on the training rows; the resulting dict is reused to clean val/test. If `rapidfuzz`
or `emoji`/`bs4` are missing, rich cleaning degrades to an empty map / light cleaning and the
notebook keeps running.

In [6]:
try:
    import emoji
    from bs4 import BeautifulSoup
    from rapidfuzz import process, fuzz
    from rapidfuzz.distance import Levenshtein
    HAVE_RICH = True
except ImportError as e:
    HAVE_RICH = False
    print(f"[info] rich deps missing ({e}); rich rows fall back to light. "
          "Enable with: pip install --user emoji beautifulsoup4 rapidfuzz")

def build_correction_map(texts, threshold=95, max_dist=2, max_singletons=20000):
    all_words = []
    for t in texts.astype(str):
        all_words.extend(re.findall(r"\b\w+\b", t.lower()))
    counts = Counter(all_words)
    vocab  = {w for w, c in counts.items() if c > 1}
    stable = [w for w in vocab if len(w) > 3 and w.isalpha()]
    singles= [w for w, c in counts.items()
              if c == 1 and len(w) > 2 and w.isalpha() and w not in vocab][:max_singletons]
    cmap = {}
    for w in singles:
        m = process.extractOne(w, stable, scorer=fuzz.ratio)
        if m and m[1] >= threshold and Levenshtein.distance(w, m[0]) <= max_dist and len(m[0]) > 3:
            cmap[w] = m[0]
    return cmap

train_mask = data["split"] == "train"

correction_map = {}
if HAVE_RICH:
    correction_map = build_correction_map(data.loc[train_mask, RAW_COL])
    print(f"correction map entries (train only): {len(correction_map):,}")

pre_light = RouteAPreprocessor(mode="light")
pre_rich  = RouteAPreprocessor(mode="rich", correction_map=correction_map, stop=STOP_RICH) \
            if HAVE_RICH else pre_light

data["clean_light"] = pre_light.transform(data["clean_eda"])      # light works on already-clean text
data["clean_rich"]  = pre_rich.transform(data[RAW_COL])           # rich needs the RAW text

# drop docs that became empty under EITHER policy so every config sees the same rows
empty = (data["clean_light"].str.len() == 0) | (data["clean_rich"].str.len() == 0)
print(f"empty after cleaning: {int(empty.sum())} dropped")
data = data[~empty].reset_index(drop=True)

train_df = data[data["split"] == "train"].reset_index(drop=True)
val_df   = data[data["split"] == "val"].reset_index(drop=True)
test_df  = data[data["split"] == "test"].reset_index(drop=True)
y_train  = train_df["sentiment"].to_numpy()
y_val    = val_df["sentiment"].to_numpy()
y_test   = test_df["sentiment"].to_numpy()

# show the two cleanings differ
ex = train_df.iloc[0]
print("\nRAW  :", str(ex[RAW_COL])[:110])
print("light:", ex["clean_light"][:110])
print("rich :", ex["clean_rich"][:110])

correction map entries (train only): 298
empty after cleaning: 14 dropped

RAW  : the cabinet dot were all detached from backing... got me
light: the cabinet dot were all detached from backing... got me
rich : cabinet dot detached backing got


### 5/ Encoders and classifiers

Three vectorizer families — **BoW** (raw counts), **One-Hot** (binary presence), **TF-IDF**.
The token pattern differs by cleaning: light text is letters-only and lowercased; rich text
carries `URL`/`MNTN` and ALL-CAPS tokens, so its vectorizers take whitespace tokens verbatim
(`lowercase=False`, `token_pattern=r"[^ ]+"`).

In [7]:
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.naive_bayes import ComplementNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.experimental import enable_halving_search_cv  # noqa: F401  (required import)
from sklearn.model_selection import HalvingGridSearchCV
from joblib import Memory
import tempfile, atexit, shutil

_CACHE_DIR = tempfile.mkdtemp(prefix="rtA_cache_")
_MEMORY    = Memory(location=_CACHE_DIR, verbose=0)
atexit.register(lambda: shutil.rmtree(_CACHE_DIR, ignore_errors=True))

CV5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Candidate Neutral upweightings, tuned by the grid search per config.
# None / "balanced" are the references; the dicts upweight only the Neutral (0) class.
NEUTRAL_WEIGHTS = [None, "balanced",
                  {-1: 1.0, 0: 1.5, 1: 1.0}]
# RandomForest also accepts the per-bootstrap variant.
RF_WEIGHTS = [None, "balanced",
              {-1: 1.0, 0: 1.5, 1: 1.0}]

def _registry(clf_name):
    # returns: estimator, clf-param grid (the "C" analogue + class_weight), needs_dense, k grid
    # NB: ComplementNB and KNN do NOT support class_weight, so they get no weighting dial.
    if clf_name == "complementNB":
        return ComplementNB(), {"clf__alpha": [0.1, 0.3, 1.0, 3.0]}, False, ["all"]
    if clf_name == "logreg":
        return (LogisticRegression(max_iter=1000),                 # no n_jobs here
                {"clf__C": [0.1, 0.5, 1.0, 2.0],
                 "clf__class_weight": NEUTRAL_WEIGHTS}, False, ["all"])
    if clf_name == "linearSVC":
        return (LinearSVC(dual="auto", max_iter=5000),
                {"clf__C": [0.1, 0.3, 0.5, 1.0],
                 "clf__class_weight": NEUTRAL_WEIGHTS}, False, ["all"])
    if clf_name == "rf":
        return (RandomForestClassifier(n_estimators=200, random_state=SEED),   # no n_jobs here
                {"clf__max_depth": [None, 20, 40],
                 "clf__class_weight": RF_WEIGHTS}, False, [3000])
    if clf_name == "knn":
        return (KNeighborsClassifier(),                            # no n_jobs here
                {"clf__n_neighbors": [15, 25, 45]}, False, [1500, 3000])
    raise ValueError(f"unknown classifier {clf_name}")

def build_vectorizer(kind, rich, ngram_range=(1, 2), min_df=10, max_features=30000):
    tp = r"[^ ]+" if rich else r"[a-z]+"
    lc = not rich
    common = dict(lowercase=lc, token_pattern=tp, ngram_range=ngram_range,
                  min_df=min_df, max_features=max_features)
    if kind == "bow":
        return CountVectorizer(**common)
    if kind == "onehot":
        return CountVectorizer(binary=True, **common)
    if kind == "tfidf":
        return TfidfVectorizer(sublinear_tf=True, **common)
    raise ValueError(kind)
    
def to_dense(x):
    """Top-level (pickle-safe) densifier for the tree model."""
    return x.toarray() if hasattr(x, "toarray") else x

def tune_cell(pp_name, vk, clf_name, verbose=True):
    col  = "clean_rich" if pp_name == "rich" else "clean_light"
    rich = (pp_name == "rich")
    est, clf_grid, needs_dense, k_grid = _registry(clf_name)

    steps = [("vec", build_vectorizer(vk, rich)), ("sel", SelectKBest(chi2))]
    if needs_dense:
        steps.append(("dense", FunctionTransformer(to_dense, accept_sparse=True)))
    steps.append(("clf", est))
    pipe = Pipeline(steps, memory=_MEMORY)   # <-- cache vec/sel across param combos

    param_grid = {"sel__k": k_grid, **clf_grid}

    gs = HalvingGridSearchCV(
        pipe, param_grid, scoring="f1_macro", cv=CV5,
        factor=3, random_state=SEED, n_jobs=-1, error_score="raise", refit=True)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=ConvergenceWarning)
        warnings.simplefilter("ignore", category=FutureWarning)
        gs.fit(train_df[col], y_train)

    return {"preprocessing": pp_name, "vectorizer": vk, "classifier": clf_name,
            "cv_macroF1": round(gs.best_score_, 4), "best_params": gs.best_params_,
            "estimator": gs.best_estimator_}

def print_result(r):
    print(f"{r['preprocessing']:5} | {r['vectorizer']:6} | {r['classifier']:13} -> F1={r['cv_macroF1']:.4f} | {r['best_params']}")


In [23]:
r01 = tune_cell("light", "bow",    "complementNB")
print_result(r01)

light | bow    | complementNB  -> F1=0.6534 | {'clf__alpha': 1.0, 'sel__k': 'all'}


In [24]:
r02 = tune_cell("light", "onehot", "complementNB")
print_result(r02)

light | onehot | complementNB  -> F1=0.6546 | {'clf__alpha': 0.1, 'sel__k': 'all'}


In [25]:
r03 = tune_cell("light", "tfidf",  "complementNB")
print_result(r03)

light | tfidf  | complementNB  -> F1=0.6528 | {'clf__alpha': 1.0, 'sel__k': 'all'}


In [27]:
r04 = tune_cell("light", "bow",    "logreg")
print_result(r04)

light | bow    | logreg        -> F1=0.6814 | {'clf__C': 0.1, 'clf__class_weight': 'balanced', 'sel__k': 'all'}


In [8]:
r05 = tune_cell("light", "onehot", "logreg")
print_result(r05)

light | onehot | logreg        -> F1=0.6816 | {'clf__C': 0.1, 'clf__class_weight': 'balanced', 'sel__k': 'all'}


In [9]:
r06 = tune_cell("light", "tfidf",  "logreg")
print_result(r06)

light | tfidf  | logreg        -> F1=0.6893 | {'clf__C': 0.5, 'clf__class_weight': 'balanced', 'sel__k': 'all'}


In [10]:
r07 = tune_cell("light", "bow",    "linearSVC")
print_result(r07)

light | bow    | linearSVC     -> F1=0.6626 | {'clf__C': 0.1, 'clf__class_weight': 'balanced', 'sel__k': 'all'}


In [11]:
r08 = tune_cell("light", "onehot", "linearSVC")
print_result(r08)

light | onehot | linearSVC     -> F1=0.6628 | {'clf__C': 0.1, 'clf__class_weight': 'balanced', 'sel__k': 'all'}


In [12]:
r09 = tune_cell("light", "tfidf",  "linearSVC")
print_result(r09)

light | tfidf  | linearSVC     -> F1=0.6866 | {'clf__C': 0.1, 'clf__class_weight': 'balanced', 'sel__k': 'all'}


In [13]:
r12 = tune_cell("rich",  "bow",    "complementNB")
print_result(r12)

rich  | bow    | complementNB  -> F1=0.6456 | {'clf__alpha': 1.0, 'sel__k': 'all'}


In [14]:
r13 = tune_cell("rich",  "onehot", "complementNB")
print_result(r13)

rich  | onehot | complementNB  -> F1=0.6464 | {'clf__alpha': 1.0, 'sel__k': 'all'}


In [15]:
r14 = tune_cell("rich",  "tfidf",  "complementNB")
print_result(r14)

rich  | tfidf  | complementNB  -> F1=0.6441 | {'clf__alpha': 1.0, 'sel__k': 'all'}


In [16]:
r15 = tune_cell("rich",  "bow",    "logreg")
print_result(r15)

rich  | bow    | logreg        -> F1=0.6756 | {'clf__C': 0.1, 'clf__class_weight': 'balanced', 'sel__k': 'all'}


In [17]:
r16 = tune_cell("rich",  "onehot", "logreg")
print_result(r16)

rich  | onehot | logreg        -> F1=0.6764 | {'clf__C': 0.1, 'clf__class_weight': 'balanced', 'sel__k': 'all'}


In [18]:
r17 = tune_cell("rich",  "tfidf",  "logreg")
print_result(r17)

rich  | tfidf  | logreg        -> F1=0.6801 | {'clf__C': 0.5, 'clf__class_weight': 'balanced', 'sel__k': 'all'}


In [19]:
r18 = tune_cell("rich",  "bow",    "linearSVC")
print_result(r18)

rich  | bow    | linearSVC     -> F1=0.6594 | {'clf__C': 0.1, 'clf__class_weight': 'balanced', 'sel__k': 'all'}


In [20]:
r19 = tune_cell("rich",  "onehot", "linearSVC")
print_result(r19)

rich  | onehot | linearSVC     -> F1=0.6601 | {'clf__C': 0.1, 'clf__class_weight': 'balanced', 'sel__k': 'all'}


In [21]:
r20 = tune_cell("rich",  "tfidf",  "linearSVC")
print_result(r20)

rich  | tfidf  | linearSVC     -> F1=0.6767 | {'clf__C': 0.1, 'clf__class_weight': {-1: 1.0, 0: 1.5, 1: 1.0}, 'sel__k': 'all'}


In [28]:
done = [r for r in [r01,r02,r03,r04,r05,r06,r07,r08,r09,
                    r12,r13,r14,r15,r16,r17,r18,r19,r20] if 'r'+f"{0}" or True]
leaderboard = (pd.DataFrame([{k:v for k,v in d.items() if k!='estimator'} for d in done])
               .sort_values("cv_macroF1", ascending=False).reset_index(drop=True))
print(leaderboard.to_string(index=False))

best_row = leaderboard.iloc[0]
print("\nWinner:", best_row.to_dict())

preprocessing vectorizer   classifier  cv_macroF1                                                                      best_params
        light      tfidf       logreg      0.6893                {'clf__C': 0.5, 'clf__class_weight': 'balanced', 'sel__k': 'all'}
        light      tfidf    linearSVC      0.6866                {'clf__C': 0.1, 'clf__class_weight': 'balanced', 'sel__k': 'all'}
        light     onehot       logreg      0.6816                {'clf__C': 0.1, 'clf__class_weight': 'balanced', 'sel__k': 'all'}
        light        bow       logreg      0.6814                {'clf__C': 0.1, 'clf__class_weight': 'balanced', 'sel__k': 'all'}
         rich      tfidf       logreg      0.6801                {'clf__C': 0.5, 'clf__class_weight': 'balanced', 'sel__k': 'all'}
         rich      tfidf    linearSVC      0.6767 {'clf__C': 0.1, 'clf__class_weight': {-1: 1.0, 0: 1.5, 1: 1.0}, 'sel__k': 'all'}
         rich     onehot       logreg      0.6764                {'clf__C': 0.1, 'c

The leaderboard answers your question directly: it isolates the effect of (a) the encoder
family, (b) the classifier, and (c) light vs extensive preprocessing, all on identical folds.
Read the `light` vs `rich` rows for the *same* vectorizer+classifier to see whether the
extensive pipeline is worth its deployment cost. TF-IDF is expected to top BoW/one-hot, but
now you have the numbers rather than the assumption.

### 7/ Source prediction (Review vs Tweet) — diagnostic only

The deployed sentiment model never sees the source: the private test set has no `Type`.
But Reviews and Tweets are near-trivially separable from text alone, so we can *recover* the
source and still report a per-domain sentiment breakdown. This section (1) trains and scores
a source classifier, then (2) uses its predictions to validate that the per-domain sentiment
numbers we'd report are reliable even without the true label.

In [29]:
# (1) source classifier: word+char TF-IDF -> LinearSVC, fit on train, evaluated on test
src_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True,
                          token_pattern=r"[a-z]+", lowercase=True)
X_src_tr = src_vec.fit_transform(train_df["clean_light"])
src_clf  = LinearSVC(C=1.0, random_state=SEED).fit(X_src_tr, train_df["Type"])

src_pred_test = src_clf.predict(src_vec.transform(test_df["clean_light"]))
src_acc = (src_pred_test == test_df["Type"].to_numpy()).mean()
print(f"Source classifier test accuracy: {src_acc:.4f}")
print(f"Source macro-F1: {f1_score(test_df['Type'], src_pred_test, average='macro'):.4f}")
print(confusion_matrix(test_df["Type"], src_pred_test, labels=["Review", "Tweet"]),
      "  (rows=true, cols=pred; order Review, Tweet)")

Source classifier test accuracy: 0.9970
Source macro-F1: 0.9952
[[30692    57]
 [   59  7454]]   (rows=true, cols=pred; order Review, Tweet)


### 8/ Final sentiment pipeline, test metrics, and per-domain validation

Rebuild the winning leaderboard configuration, refit on **train + val**, evaluate once on
the held-out **test** set, then break the test macro-F1 down by **true** source and by
**predicted** source. If the two breakdowns agree, the per-domain story holds up under the
real deployment condition (no `Type` available).

In [31]:
from sklearn.base import clone
best_pp     = best_row["preprocessing"]
best_vk     = best_row["vectorizer"]
best_clf    = best_row["classifier"]
best_params = best_row["best_params"]

_k = best_params["sel__k"]
best_cap = _k if _k == "all" else int(_k)      # <-- was int(best_params["sel__k"]); int("all") crashes
clf_params = {k.split("clf__", 1)[1]: v
              for k, v in best_params.items() if k.startswith("clf__")}

est, _, needs_dense, _ = _registry(best_clf)
est = clone(est).set_params(**clf_params)

steps = [
    ("pre", RouteAPreprocessor(mode=best_pp, correction_map=correction_map, stop=STOP_RICH)
            if best_pp == "rich" else RouteAPreprocessor(mode="light")),
    ("vec", build_vectorizer(best_vk, best_pp == "rich")),
    ("sel", SelectKBest(chi2, k=best_cap)),
]
if needs_dense:
    steps.append(("dense", FunctionTransformer(to_dense, accept_sparse=True)))
steps.append(("clf", est))
final_pipeline = Pipeline(steps)

# refit on train+val using RAW text (the 'pre' step does the cleaning end to end)
X_trval = pd.concat([train_df[RAW_COL], val_df[RAW_COL]], ignore_index=True)
y_trval = np.concatenate([y_train, y_val])
final_pipeline.fit(X_trval, y_trval)

y_pred = final_pipeline.predict(test_df[RAW_COL])
print(f"Winning config: {best_pp} | {best_vk} | {best_clf}")
print(f"TEST macro-F1: {f1_score(y_test, y_pred, average='macro'):.4f}\n")
print(classification_report(y_test, y_pred, labels=[-1, 0, 1],
      target_names=["Negative", "Neutral", "Positive"]))
print("Confusion matrix (rows=true, cols=pred) order [-1,0,1]:")
print(confusion_matrix(y_test, y_pred, labels=[-1, 0, 1]))

print("\nPer-domain macro-F1 — TRUE source vs PREDICTED source:")
true_type = test_df["Type"].to_numpy()
for dom in ["Review", "Tweet"]:
    m_true = true_type == dom
    m_pred = src_pred_test == dom
    f_true = f1_score(y_test[m_true], y_pred[m_true], average="macro") if m_true.any() else float("nan")
    f_pred = f1_score(y_test[m_pred], y_pred[m_pred], average="macro") if m_pred.any() else float("nan")
    print(f"  {dom:>7}: true-source {f_true:.4f} (n={m_true.sum():,})  |  "
          f"predicted-source {f_pred:.4f} (n={m_pred.sum():,})")

Winning config: light | tfidf | logreg
TEST macro-F1: 0.6969

              precision    recall  f1-score   support

    Negative       0.77      0.75      0.76     13468
     Neutral       0.50      0.59      0.54      9528
    Positive       0.83      0.76      0.79     15266

    accuracy                           0.71     38262
   macro avg       0.70      0.70      0.70     38262
weighted avg       0.73      0.71      0.72     38262

Confusion matrix (rows=true, cols=pred) order [-1,0,1]:
[[10067  2732   669]
 [ 2235  5614  1679]
 [  806  2899 11561]]

Per-domain macro-F1 — TRUE source vs PREDICTED source:
   Review: true-source 0.6837 (n=30,749)  |  predicted-source 0.6836 (n=30,751)
    Tweet: true-source 0.5640 (n=7,513)  |  predicted-source 0.5664 (n=7,511)
